### Double-check the entries in your final Lupus table cf. the literature.

In [1]:
import re
import numpy  as np
import pandas as pd
pd.set_option('display.max_columns', None)

`1:` Load your table and cross match with Luhman's (2022) tables (stars and discs).
- In stars, check: "SourceID_2MASS", "SourceID_WISE", "SourceID_GaiaDR3", "SpT_L22", "kinpopL22", "pospopL22";
- In discs, check: "SourceID_2MASS", "SourceID_WISE", "discClassL22".

In [2]:
# Load your extended version of the table:
lupus_mine = pd.read_csv(
    "lupus_vs_polnitzky.dat",
    skiprows = 49,
    sep      = r'\s+',
    names   = [
        "Region", "Name", "2MASS", "SourceID_2MASS", "WISE", "SourceID_WISE", "GaiaDR3", "SourceID_GaiaDR3",
        "SpT_E26", "SpT_L22", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22", 
        "kinpopL22", "pospopL22", "memK_L22", "memKP_L22", "mem_T22", "discClassL22", "COsizeS21?", "comments"
    ]
)
print(len(lupus_mine))
# Numbers match: 100, yes!

98


In [3]:
# Load Luhman22a's table:
luhman_stars = pd.read_fwf(
    "table1_luhman2022a_orig.dat",
    names  = [
        "SourceID_GaiaDR3", "SourceID_GaiaDR2", "NameOther",
        "RAdeg", "DEdeg",
        "SpType", "r_SpType", "AdoptSpType",
        "EWLi", "r_EWLi",
        "pmRA", "e_pmRA", "pmDec", "e_pmDec", "plx", "e_plx",
        "r_med_geo", "r_lo_geo", "r_hi_geo",
        "RVel", "e_RVel", "r_RVel",
        "U U", "e_U", "V V", "e_V", "W W", "e_W",
        "Gmag G", "e_Gmag", "GBPmag", "e_GBPmag", "GRPmag", "e_GRPmag",
        "RUWE", "kin", "pos",
        "2MASS", "SourceID_2MASS_L22", "2MASS_angsep", "2MASS_closest",
        "WISE", "SourceID_WISE_L22", "WISE_angsep", "WISE_closest"
    ],
    skiprows = 273,
)
print(len(luhman_stars))
# Numbers match: 10509, yes!

10509


In [4]:
# Load Luhman22b's table:
luhman_discs = pd.read_fwf(
    "table1_luhman2022b_orig.dat",
    names  = [
        "WISE", "SourceID_WISE_L22", "RAdeg","DEdeg", "2MASS", "SourceID_2MASS", "2MASS_angsep",
        "Jmag 2MASS", "e_Jmag", "Hmag 2MASS", "e_Hmag", "Ksmag 2MASS", "e_Ksmag",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "f_W3mag", "W4mag", "e_W4mag", "f_W4mag",
        "ExcW2 Excess present in W2?", "ExcW3 Excess present in W3?", "ExcW4 Excess present in W4?", "DiskType Disk Typec"
    ],
    skiprows = 45,
)
print(len(luhman_discs))
# Numbers match: 9675, yes!

9675


In [5]:
# Using your 2MASS ID match the number of stars with disc in Luhman (2022a).
# There are some duplicates (sources with same 2MASS ID but different Gaia ID, see below). Trim by GaiaDR3 ID and check 2MASS and WISE:
matched_stars = lupus_mine.merge(
    luhman_stars[["SourceID_GaiaDR3", "SourceID_2MASS_L22", "2MASS_closest", "SourceID_WISE_L22", "WISE_closest", "AdoptSpType", "kin", "pos"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_stars))

# Using your 2MASS ID match the number of stars with disc in Luhman (2022b):
matched_discs = lupus_mine.merge(
    luhman_discs[["SourceID_2MASS", "SourceID_WISE_L22", "DiskType Disk Typec"]],
    on  = "SourceID_2MASS",
    how = "left"
)
print(len(matched_discs))

98
98


In [6]:
# Check that the 2MASS ID in my table and Luhman match:
# Some Gaia sources in my table are not in Luhman2022. 
# Other sources with 2MASS ID in my table lack a Gaia ID and are this absent from Luhman's table.
for i in range(len(matched_stars["SourceID_GaiaDR3"])):
    if matched_stars["SourceID_2MASS"][i] != matched_stars["SourceID_2MASS_L22"][i]: 
        print(
            matched_stars["SourceID_GaiaDR3"][i],
            matched_stars["SourceID_2MASS"][i], matched_stars["SourceID_2MASS_L22"][i], matched_stars["2MASS_closest"][i], 
            matched_stars["memK_L22"][i]
        )

-99 J15430131-3409153 nan nan -
-99 J15430227-3444059 nan nan -
-99 J15450634-3417378 nan nan -
-99 J16011549-4152351 nan nan -
-99 J16032939-4140018 nan nan -
5997077202094925184 J16070384-3911113 nan nan -
5997076721058575360 J16070854-3914075 nan nan -
5997077167735183872 J16071007-3911033 nan nan -
-99 J16075475-3915446 nan nan -
-99 J16080175-3912316 nan nan -
5997087746218718720 J16081497-3857145 nan nan -
5997033015470255488 -99 nan nan N
5997083008890845184 J16082972-3903110 nan nan -
-99 -99 nan nan -
5997082218616859264 -99 nan nan N
5997033290348155392 J16085373-3914367 nan nan -
-99 J16085828-3907355 nan nan -
-99 J16085834-3907491 nan nan -
-99 J16091644-3904438 nan nan -
-99 J16092032-3904015 nan nan -
-99 J16092317-3904074 nan nan -
-99 -99 nan nan -
-99 J16093928-3904316 nan nan -
5997407330454214016 J16100133-3906449 nan nan -
5997402103466530176 J16102741-3902299 nan nan -
5997600737113961472 J16120445-3809589 nan nan -
-99 -99 nan nan -


In [7]:
# Same for WISE:
# 5997021981682416256 genuinely lack a WISEA ID.
for i in range(len(matched_stars["SourceID_GaiaDR3"])):
    if matched_stars["SourceID_WISE"][i] != matched_stars["SourceID_WISE_L22"][i]: 
        print(
            matched_stars["SourceID_GaiaDR3"][i],
            matched_stars["SourceID_WISE"][i],matched_stars["SourceID_WISE_L22"][i], matched_stars["WISE_closest"][i], 
            matched_stars["memK_L22"][i]
        )

-99 -99 nan nan -
-99 -99 nan nan -
-99 J154506.33-341738.1 nan nan -
-99 J160115.49-415235.3 nan nan -
-99 -99 nan nan -
5997077202094925184 J160703.82-391111.6 nan nan -
5997076721058575360 J160708.54-391407.7 nan nan -
5997077167735183872 J160710.05-391103.5 nan nan -
-99 J160754.72-391544.5 nan nan -
-99 -99 nan nan -
5997087746218718720 -99 nan nan -
5997033015470255488 J160828.09-391309.9 J160828.08-391310.0 Y N
5997083008890845184 J160829.72-390311.1 nan nan -
-99 -99 nan nan -
5997082218616859264 -99 nan nan N
5997033290348155392 -99 nan nan -
-99 -99 nan nan -
-99 -99 nan nan -
-99 -99 nan nan -
-99 -99 nan nan -
-99 -99 nan nan -
-99 - nan nan -
-99 - nan nan -
5997407330454214016 -99 nan nan -
5997402103466530176 J161027.43-390229.8 nan nan -
5997600737113961472 -99 nan nan -
5997021981682416256 -99 nan nan Y
-99 -99 nan nan -


In [8]:
# Compare SpTs (no differences)
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["SpT_L22"][i] != matched_stars["AdoptSpType"][i]: 
        print(
            matched_stars["SourceID_2MASS"][i], 
            matched_stars["AdoptSpType"][i], matched_stars["SpT_L22"][i], matched_stars["kin"][i]
        )

J15430131-3409153 nan - nan
J15430227-3444059 nan - nan
J15450634-3417378 nan - nan
J16011549-4152351 nan - nan
J16032939-4140018 nan - nan
J16070384-3911113 nan - nan
J16070854-3914075 nan - nan
J16071007-3911033 nan - nan
J16075475-3915446 nan - nan
J16080175-3912316 nan - nan
J16081497-3857145 nan - nan
J16082972-3903110 nan - nan
-99 nan - nan
J16085373-3914367 nan - nan
J16085828-3907355 nan - nan
J16085834-3907491 nan - nan
J16091644-3904438 nan - nan
J16092032-3904015 nan - nan
J16092317-3904074 nan - nan
-99 nan - nan
J16093928-3904316 nan - nan
J16100133-3906449 nan - nan
J16102741-3902299 nan - nan
J16120445-3809589 nan - nan
-99 nan - nan


In [9]:
# Compare the kinematic populations (no differences):
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["kinpopL22"][i] != matched_stars["kin"][i]: 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["kinpopL22"][i], matched_stars["kin"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J16011549-4152351 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16070854-3914075 - nan
J16071007-3911033 - nan
J16075475-3915446 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082972-3903110 - nan
-99 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
-99 - nan


In [10]:
# Compare the kinematic populations (no differences):
for i in range(len(matched_stars["SourceID_2MASS"])):
    if matched_stars["kinpopL22"][i] != matched_stars["kin"][i]: 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["kinpopL22"][i], matched_stars["kin"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J16011549-4152351 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16070854-3914075 - nan
J16071007-3911033 - nan
J16075475-3915446 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082972-3903110 - nan
-99 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
-99 - nan


In [11]:
# Check that the WISE IDs are the same:
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["SourceID_WISE"][i] != matched_discs["SourceID_WISE_L22"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["SourceID_GaiaDR3"][i], matched_discs["SourceID_WISE"][i], matched_discs["SourceID_WISE_L22"][i])

J15430131-3409153 -99 -99 nan
J15430227-3444059 -99 -99 nan
J15450634-3417378 -99 J154506.33-341738.1 nan
J16011549-4152351 -99 J160115.49-415235.3 nan
J16032939-4140018 -99 -99 nan
J16070384-3911113 5997077202094925184 J160703.82-391111.6 nan
J16070854-3914075 5997076721058575360 J160708.54-391407.7 nan
J16071007-3911033 5997077167735183872 J160710.05-391103.5 nan
J16075475-3915446 -99 J160754.72-391544.5 nan
J16080175-3912316 -99 -99 nan
J16081497-3857145 5997087746218718720 -99 nan
-99 5997033015470255488 J160828.09-391309.9 nan
J16082972-3903110 5997083008890845184 J160829.72-390311.1 nan
-99 -99 -99 nan
-99 5997082218616859264 -99 nan
J16085373-3914367 5997033290348155392 -99 nan
J16085828-3907355 -99 -99 nan
J16085834-3907491 -99 -99 nan
J16091644-3904438 -99 -99 nan
J16092032-3904015 -99 -99 nan
J16092317-3904074 -99 -99 nan
-99 -99 - nan
J16093928-3904316 -99 - nan
J16100133-3906449 5997407330454214016 -99 nan
J16102741-3902299 5997402103466530176 J161027.43-390229.8 nan
J16120

In [12]:
# Check that the disc classification is the same:
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["discClassL22"][i] != matched_discs["DiskType Disk Typec"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["DiskType Disk Typec"][i], matched_discs["discClassL22"][i])

J15430131-3409153 nan -
J15430227-3444059 nan -
J15450634-3417378 nan -
J16011549-4152351 nan -
J16032939-4140018 nan -
J16070384-3911113 nan -
J16070854-3914075 nan -
J16071007-3911033 nan -
J16075475-3915446 nan -
J16080175-3912316 nan -
J16081497-3857145 nan -
-99 nan -
J16082972-3903110 nan -
-99 nan -
-99 nan -
J16085373-3914367 nan -
J16085828-3907355 nan -
J16085834-3907491 nan -
J16091644-3904438 nan -
J16092032-3904015 nan -
J16092317-3904074 nan -
-99 nan -
J16093928-3904316 nan -
J16100133-3906449 nan -
J16102741-3902299 nan -
J16120445-3809589 nan -
J16134410-3736462 nan -
J16083733-3923109 nan -
-99 nan -


`2:` Check that your 2MASS designations are the cosest to the identified _Gaia_ members:

In [13]:
# Match the Luhman2022's catalogue by 2MASS ID:
matched_stars2MASS = lupus_mine.merge(
    luhman_stars,
    left_on  = "SourceID_2MASS",
    right_on = "SourceID_2MASS_L22",
    how = "left"
)
print(len(matched_stars2MASS))

# Print number of duplicates:
dups = matched_stars2MASS[matched_stars2MASS["SourceID_2MASS"].duplicated(keep=False)]
print(len(dups))
# print(dups["SourceID_2MASS"].tolist())
# Good, did pick the closest!
for name in np.unique(dups["SourceID_2MASS"].tolist()):
    if name != '-99':
        mask = matched_stars["SourceID_2MASS"] == name
        print(name, matched_stars["2MASS_closest"][mask])

101
11
J15451286-3417305 7    Y
Name: 2MASS_closest, dtype: object
J15555030-3801329 16    Y
Name: 2MASS_closest, dtype: object
J16070061-3902194 33    Y
Name: 2MASS_closest, dtype: object


`3:` Check that your KP membership classification is correct.

In [14]:
lupus_mine_refined = lupus_mine[
    lupus_mine["kinpopL22"].astype(str).str.contains("l", na=False) & 
    lupus_mine["pospopL22"].astype(str).str.contains("l", na=False)
]
print("KP Members:", len(lupus_mine_refined))
for i in range(len(lupus_mine_refined)):
    print(lupus_mine_refined["SourceID_2MASS"].tolist()[i], lupus_mine_refined["memKP_L22"].tolist()[i])

KP Members: 57
J15392776-3446171 Y
J15392828-3446180 Y
J15450887-3417333 Y
J15451286-3417305 Y
J15451741-3418283 Y
J15464473-3430354 Y
J15475062-3528353 Y
J15475693-3514346 Y
J15491210-3539051 Y
J15493074-3549514 Y
J15555030-3801329 Y
J15560921-3756057 Y
J15564230-3749154 Y
J15580252-3736026 Y
J15591647-4157102 Y
J16000060-4221567 Y
J16000236-4222145 Y
J16002612-4153553 Y
J16003103-4143369 Y
J16004452-4155310 Y
J16004943-4130038 Y
J16070061-3902194 Y
J16071159-3903475 Y
J16073773-3921388 Y
J16075230-3858059 Y
J16080017-3902595 Y
J16081263-3908334 Y
J16082180-3904214 Y
J16082249-3904464 Y
J16082404-3905494 Y
J16083026-3906111 Y
J16083081-3905488 Y
J16083427-3906181 Y
J16083617-3923024 Y
J16083976-3906253 Y
J16084940-3905393 Y
J16085157-3903177 Y
J16085324-3914401 Y
J16085468-3937431 Y
J16085529-3848481 Y
J16085553-3902339 Y
J16085780-3902227 Y
J16090141-3925119 Y
J16090185-3905124 Y
J16090621-3908518 Y
J16092697-3836269 Y
J16094434-3913301 Y
J16094864-3911169 Y
J16095628-3859518 Y
J1610

In [15]:
lupus_mine_wrong = lupus_mine[
    ~(lupus_mine["kinpopL22"].astype(str).str.contains("l", na=False) & lupus_mine["pospopL22"].astype(str).str.contains("l", na=False))
]
print("KP NON-Members:",len(lupus_mine_wrong))
for i in range(len(lupus_mine_wrong)):
    print(lupus_mine_wrong["SourceID_2MASS"].tolist()[i], lupus_mine_wrong["memKP_L22"].tolist()[i])

KP NON-Members: 41
J15430131-3409153 -
J15430227-3444059 -
J15445789-3423392 N
J15450634-3417378 -
J15480523-3515526 N
J15514695-3556440 Y
J15560210-3655282 N
J15592523-4235066 N
J15592838-4021513 N
J16011549-4152351 -
J16030548-4018254 N
J16032939-4140018 -
J16070384-3911113 -
J16070854-3914075 -
J16071007-3911033 Y
J16075475-3915446 -
J16080175-3912316 -
J16081497-3857145 -
J16082576-3906011 N
-99 N
J16082972-3903110 -
J16083070-3828268 N
-99 -
-99 N
J16085143-3905304 N
J16085373-3914367 -
J16085828-3907355 -
J16085834-3907491 -
J16091644-3904438 -
J16092032-3904015 -
J16092317-3904074 -
-99 -
J16093928-3904316 -
J16100133-3906449 -
J16101307-3846165 N
J16102741-3902299 -
J16120445-3809589 -
J16124373-3815031 N
J16134410-3736462 N
J15414081-3345188 N
-99 -


`3:` Import Ratzenbock's and Polnitzki's tables and check that the cluster membership and disc status are correct.
- Check "SourceID_WISE", "SourceID_GaiaDR3", "SigMACluster", "Disc?CCD", "Disc?SED", "Disc?L22"

In [16]:
# Load Ratzenbock's table:
ratzenbock = pd.read_csv(
    "tableE1_ratzenbock2023a.dat",
    header = None,
    names  = [
        "SourceID_GaiaDR3", "SigMA_membership", "Stability", "Distance (pc)", "e_d_upper (pc)", "e_d_lower (pc)",
        "X (pc)", "Y (pc)", "Z (pc)", "e_X_upper (pc)", "e_Y_upper (pc)", "e_Z_upper (pc)", "e_X_lower (pc)", "e_Y_lower (pc)", "e_Z_lower (pc)",
        "v_alpha (km s−1)", "v_delta (km s−1)", "e_v_alpha_upper (km s−1)", "e_v_delta_upper (km s−1)", "e_v_alpha_lower (km s−1)", "e_v_delta_lower (km s−1)",
        "v_alpha_LSR (km s−1)", "v_delta_LSR (km s−1)", "v_ERV (km s-1)",
    ],
    sep    = r"\s+",
)
print(len(ratzenbock))
# Numbers match (caption table E1): 13103.

13103


In [17]:
# Load Polnitky2026's table:
polnitzky = pd.read_fwf(
    "tableC2_polnitzky2026.dat",
    skiprows = 1,
    names  = [
        "SourceID_GaiaDR3", "RA (deg)", "Dec (deg)", "l (degree)", "b (degree)", "RUWE", "SourceID_AllWISE",
        "W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "W4mag", "e_W4mag", "Jmag", "e_Jmag", "Hmag", "e_Hmag", "Kmag", "e_Kmag",
        "snr1", "chi2W1", "snr2", "chi2W2", "snr3", "chi2W3", "snr4", "chi2W4", 
        "Disk_CCD", "W123", "W124", "HKW2", "HKW3", "Disk_SED",
        "alpha_KW3", "e_alpha_KW3", "intercept_KW3", "e_intercept_KW3", 
        "alpha_KW4", "e_alpha_KW4", "intercept_KW4", "e_intercept_KW4", 
        "alpha_W13", "e_alpha_W13", "intercept_W13", "e_intercept_W13", 
        "alpha_W14", "e_alpha_W14", "intercept_W14", "e_intercept_W14", 
        "alpha_KW2", "e_alpha_KW2", "intercept_KW2", "e_intercept_KW2", 
        "alpha", 
        "DiskClass_Luhman", "Disk_Luhman", "SigMA_label"
    ],
)
print(len(polnitzky))
# Numbers match (direct table inspection): 10270

10270


In [18]:
# Using GaiaDR3 IDs match your table with the Ratzenbock one to double-check the kinematic membership
matched_stars = lupus_mine.merge(
    ratzenbock[["SourceID_GaiaDR3", "SigMA_membership"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_stars))

# Using GaiaDR3 IDs match your table with the Polnitzky one to double-check the disc classification
matched_discs = lupus_mine.merge(
    polnitzky[["SourceID_GaiaDR3", "Disk_CCD", "Disk_SED", "Disk_Luhman"]],
    on  = "SourceID_GaiaDR3",
    how = "left"
)
print(len(matched_discs))

98
98


In [19]:
# Cluster membership -> agree!
for i in range(len(matched_stars["SourceID_2MASS"])):
    if str(matched_stars["SigMACluster"][i]) != str(matched_stars["SigMA_membership"][i]): 
        print(matched_stars["SourceID_2MASS"][i], matched_stars["SigMACluster"][i], matched_stars["SigMA_membership"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J15480523-3515526 - nan
J16011549-4152351 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16075475-3915446 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082576-3906011 - nan
J16082972-3903110 - nan
-99 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
-99 - nan


In [20]:
# CCD discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?CCD"][i] != matched_discs["Disk_CCD"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?CCD"][i], matched_discs["Disk_CCD"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J15480523-3515526 - nan
J15555030-3801329 - nan
J16011549-4152351 - nan
J16030548-4018254 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16075475-3915446 - nan
J16080017-3902595 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082576-3906011 - nan
-99 - nan
J16082972-3903110 - nan
J16083081-3905488 - nan
-99 - nan
-99 - nan
J16084940-3905393 - nan
J16085324-3914401 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16101984-3836065 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
J16134410-3736462 - nan
J16083733-3923109 - nan
-99 - nan


In [21]:
# SED discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?SED"][i] != matched_discs["Disk_SED"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?SED"][i], matched_discs["Disk_SED"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J15480523-3515526 - nan
J15555030-3801329 - nan
J16011549-4152351 - nan
J16030548-4018254 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16075475-3915446 - nan
J16080017-3902595 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082576-3906011 - nan
-99 - nan
J16082972-3903110 - nan
J16083081-3905488 - nan
-99 - nan
J16083427-3906181 - nan
-99 - nan
J16084940-3905393 - nan
J16085324-3914401 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16101984-3836065 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
J16134410-3736462 - nan
J16083733-3923109 - nan
-99 - nan


In [22]:
# L22 discs -> agree!
for i in range(len(matched_discs["SourceID_2MASS"])):
    if matched_discs["Disc?L22"][i] != matched_discs["Disk_Luhman"][i]: 
        print(matched_discs["SourceID_2MASS"][i], matched_discs["Disc?L22"][i], matched_discs["Disk_Luhman"][i])

J15430131-3409153 - nan
J15430227-3444059 - nan
J15450634-3417378 - nan
J15480523-3515526 - nan
J15555030-3801329 - nan
J16011549-4152351 - nan
J16030548-4018254 - nan
J16032939-4140018 - nan
J16070384-3911113 - nan
J16070854-3914075 - nan
J16071007-3911033 - nan
J16075475-3915446 - nan
J16080017-3902595 - nan
J16080175-3912316 - nan
J16081497-3857145 - nan
J16082576-3906011 - nan
-99 - nan
J16082972-3903110 - nan
J16083081-3905488 - nan
-99 - nan
-99 - nan
J16084940-3905393 - nan
J16085324-3914401 - nan
J16085373-3914367 - nan
J16085828-3907355 - nan
J16085834-3907491 - nan
J16091644-3904438 - nan
J16092032-3904015 - nan
J16092317-3904074 - nan
-99 - nan
J16093928-3904316 - nan
J16100133-3906449 - nan
J16101984-3836065 - nan
J16102741-3902299 - nan
J16120445-3809589 - nan
J16134410-3736462 - nan
J16083733-3923109 - nan
-99 - nan
